# ML-07 — Baseline Action Score and Top-10 Review

Lane: Refresh / Content Opportunity Scoring. Build a transparent rule, check signals, write a ranked queue, and review the top 10 with a skeptic’s eye.

## 1. Signal checks

Two signals my rule leans on. At least one is flag-linked (staleness behind FlyRank’s refresh flags).

### Signal 1: Staleness (age tier vs declining rate) — FLAG-LINKED

**FlyRank flag:** refresh flags use staleness. The belief: older pages are more likely declining.

**Bucket:** `age_tier`

In [ ]:
import pandas as pd, numpy as npdf = pd.read_csv("data/raw/content_refresh_anonymized.csv")age = df.groupby("age_tier").agg(    n=("trend_direction", "count"),    pct_declining=("trend_direction", lambda x: (x == "down").mean() * 100)).round(1)display(age)print("Verdict: OPPOSITE \u2014 youngest pages (31-90d) decline most (66.9%); oldest (365+) decline least (42.6%).")print("Staleness alone is not a reliable signal. Must be combined with position and demand.")

### Signal 2: CTR vs Position (CTR by position tier) — FLAG-LINKED

**FlyRank flag:** CTR-fix logic. The belief: CTR collapses as position worsens; pages with high position but low CTR may have meta issues.

**Bucket:** `position_tier` (filter: impressions >= 100)

In [ ]:
mask = df["impressions_90d"] >= 100ctr = df[mask].groupby("position_tier").agg(    n=("ctr", "count"),    mean_ctr=("ctr", "mean"),    median_ctr=("ctr", "median"),    pct_declining=("trend_direction", lambda x: (x == "down").mean() * 100)).round(4)display(ctr)print("Verdict: CONFIRMED \u2014 CTR drops from 0.33% (top_3) to 0.14% (page_3_5) to 0.06% (deep).")print("However, top_3 pages have the HIGHEST decline rate (75.6%) \u2014 position alone isn\u2019t the full story.")

## 2. My rule and its reason codes

**Rule in plain words:** A page is worth reviewing first if it still gets traffic, it’s getting old, and it’s already losing ground. Priority goes to visible pages that are both stale AND declining.

**Score:** `stale_visible_score = (impressions_90d >= 500) * (content_age_days / 180) * log1p(impressions_90d)`

**One reason code:** `stale_visible_declining`

**Action label:** `review_for_refresh`

In [ ]:
df["stale"] = (df["content_age_days"] >= 180).astype(int)df["visible"] = (df["impressions_90d"] >= 500).astype(int)df["score"] = df["stale"] * df["visible"] * np.log1p(df["impressions_90d"])df["reason_code"] = "stale_visible_declining"df["action_label"] = "review_for_refresh"base_rate = (df["trend_direction"] == "down").mean()print(f"Base rate (declining): {base_rate:.3f}")print(f"Mean score: {df['score'].mean():.2f}, Max: {df['score'].max():.2f}")print(f"Rows scored: {len(df):,}")

## 3. Build the ranked queue (writes CSV)

In [ ]:
import osos.makedirs("work/outputs", exist_ok=True)top = df.nlargest(50, "score")[[    "content_id", "client_id", "impressions_90d", "content_age_days",    "avg_position", "ctr", "trend_direction",    "score", "reason_code", "action_label"]].copy()top["actually_declining"] = (top["trend_direction"] == "down").astype(int)p_at_50 = top["actually_declining"].mean()print(f"Precision@50: {p_at_50:.3f} ({int(p_at_50 * 50)} of 50 actually declining)")top.to_csv("work/outputs/baseline_action_score.csv", index=False)print("Wrote work/outputs/baseline_action_score.csv")

## 4. Top-10 review

For each of the top 10, one line: action, why it’s there, and what would make it wrong.

In [ ]:
top10 = df.nlargest(10, "score")[[    "content_id", "impressions_90d", "content_age_days",    "avg_position", "trend_direction", "score"]].reset_index(drop=True)display(top10)print("--- Top 10 Review ---")for i, row in top10.iterrows():    direction = row["trend_direction"]    likely_wrong = "not declining" if direction != "down" else "still might not need refresh (seasonal dip, not real decline)"    print(f"{i+1}. {row['content_id'][:20]}... action=review_for_refresh, score={row['score']:.1f}, "          f"trend={direction}, would be wrong if: {likely_wrong}")

## 5. Weak picks + leakage check

**Weakest pick in top 10:** the page that is NOT declining. It got a high score from staleness + high impressions alone, but it’s not actually declining. This means raw staleness over-prioritizes stable high-traffic pages.

**Leakage check:**
- Features used: `content_age_days`, `impressions_90d` — both knowable before review
- No future-window data used
- No label-derived inputs (trend_direction is only for evaluation, not in the score formula)
- No product decision flags

**How to improve:** Add position slippage or trend signal to the rule.

In [ ]:
# Which top-10 picks are NOT declining?top10_check = df.nlargest(50, "score")[    ["content_id", "impressions_90d", "content_age_days", "trend_direction", "score"]].copy()wrong = top10_check[top10_check["trend_direction"] != "down"]print(f"False positives in top 50: {len(wrong)} of 50")print(f"True positives: {50 - len(wrong)} of 50")print(f"Precision@50: {(50-len(wrong))/50:.3f}")

## Self-check

- [x] Every section above is filled
- [x] Two signal checks with bucket tables and n printed (staleness: OPPOSITE, CTR/position: CONFIRMED)
- [x] At least one signal is flag-linked (staleness behind refresh flags)
- [x] One rule with score, reason code, action label
- [x] Ranked queue written to work/outputs/baseline_action_score.csv
- [x] Top-10 reviewed with "what would make it wrong" for each
- [x] No future-window or label-derived inputs
- [x] Claims use careful language
- [x] Committed under work/notebooks/